# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
%pip -q install duckdb huggingface_hub pandas scikit-learn

import os
import getpass
import duckdb
import pandas as pd
import numpy as np

from huggingface_hub import snapshot_download

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

warehouse_path = snapshot_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    token=HF_TOKEN,
    allow_patterns=[
        "fact_content_daily_performance/month=2026-03/*.parquet",
        "fact_content_daily_performance/month=2026-04/*.parquet",
    ],
)

con = duckdb.connect()

MARCH = f"read_parquet('{warehouse_path}/fact_content_daily_performance/month=2026-03/*.parquet')"
APRIL = f"read_parquet('{warehouse_path}/fact_content_daily_performance/month=2026-04/*.parquet')"

print("Data ready.")

Note: you may need to restart the kernel to use updated packages.


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Data ready.


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

### Feature vector

I build one feature vector per client × content item using only March 2026 data.

I use five simple search-performance features:

- total impressions
- total clicks
- CTR
- average search position
- number of active days

No categorical variables are used in this first feature vector.

In [2]:
feature_df = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS impressions,

        SUM(gsc_clicks) AS clicks,

        100.0 * SUM(gsc_clicks)
            / NULLIF(SUM(gsc_impressions), 0) AS ctr,

        AVG(NULLIF(gsc_avg_position, 0)) AS avg_position,

        COUNT(*) FILTER (
            WHERE gsc_impressions > 0
        ) AS active_days

    FROM {MARCH}

    WHERE gsc_data_available IS TRUE

    GROUP BY
        client_hash_id,
        content_hash_id

    HAVING SUM(gsc_impressions) > 0
""").df()


# Fill remaining missing average positions with the median
position_median = feature_df["avg_position"].median()

feature_df["avg_position"] = (
    feature_df["avg_position"]
    .fillna(position_median)
)

feature_df.head()

,client_hash_id,content_hash_id,impressions,clicks,ctr,avg_position,active_days
0,client_e547b89c05043229,content_36bf7800b1377cd9,5976.0,4.0,0.066934,24.241927,29
1,client_e547b89c05043229,content_a67e0e8670feeb39,1047.0,0.0,0.000000,24.684638,29
2,client_e547b89c05043229,content_d2f8156fccb183c7,1019.0,0.0,0.000000,22.826284,29
3,client_e547b89c05043229,content_31f5930e9ffb1c8b,1420.0,0.0,0.000000,28.087895,29
4,client_e547b89c05043229,content_f94d1937b8cd4ddc,728.0,0.0,0.000000,23.539806,29


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

### Feature notes

| Feature | Meaning | Missing-value handling | Available before prediction? |
|---|---|---|---|
| `impressions` | Total March search impressions | Pages with zero total impressions are excluded | Yes — only March data |
| `clicks` | Total March search clicks | Aggregated only from available GSC data | Yes — only March data |
| `ctr` | March clicks divided by March impressions | Zero-impression pages are excluded | Yes — only March data |
| `avg_position` | Average March Google search position | Missing values are filled with the median observed position | Yes — only March data |
| `active_days` | Number of March days with at least one impression | No fill needed | Yes — only March data |

All five model features are numeric. Client and content IDs are kept only as context and are not model features.

In [3]:
features = [
    "impressions",
    "clicks",
    "ctr",
    "avg_position",
    "active_days"
]

missing_check = pd.DataFrame({
    "feature": features,
    "missing_n": [
        feature_df[col].isna().sum()
        for col in features
    ],
    "missing_pct": [
        100 * feature_df[col].isna().mean()
        for col in features
    ]
})

missing_check

,feature,missing_n,missing_pct
0,impressions,0,0.0
1,clicks,0,0.0
2,ctr,0,0.0
3,avg_position,0,0.0
4,active_days,0,0.0


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

### Leakage hunt

The prediction moment is the end of March 2026.

Therefore, valid features may only use information available by the end of March.

I check for three types of leakage:

1. future information, such as April performance;
2. label-derived information;
3. existing product flags or scores that already encode a decision.

Client IDs, content IDs, URLs, and private queries are also excluded from the feature set.

In [4]:
april = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS april_impressions

    FROM {APRIL}

    WHERE gsc_data_available IS TRUE

    GROUP BY
        client_hash_id,
        content_hash_id
""").df()


model_data = feature_df.merge(
    april,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)


model_data["is_future_decline"] = (
    model_data["april_impressions"]
    < 0.8 * model_data["impressions"]
).astype(int)

model_data[
    features + ["april_impressions", "is_future_decline"]
].head()

,impressions,clicks,ctr,avg_position,active_days,april_impressions,is_future_decline
0,5976.0,4.0,0.066934,24.241927,29,4876.0,0
1,1047.0,0.0,0.000000,24.684638,29,802.0,1
2,1019.0,0.0,0.000000,22.826284,29,1338.0,0
3,1420.0,0.0,0.000000,28.087895,29,1321.0,0
4,728.0,0.0,0.000000,23.539806,29,477.0,1


### Feature-list check

The final feature list should contain no future columns, labels, IDs, URLs, queries, product flags, or existing scores.

In [5]:
forbidden_words = [
    "april",
    "future",
    "label",
    "flag",
    "score",
    "client_hash_id",
    "content_hash_id",
    "url",
    "query"
]

leak_candidates = [
    feature
    for feature in features
    if any(word in feature.lower() for word in forbidden_words)
]

print("Features:", features)
print("Suspicious features:", leak_candidates)

Features: ['impressions', 'clicks', 'ctr', 'avg_position', 'active_days']
Suspicious features: []


### Deliberate leakage test

To verify that the leakage check matters, I deliberately add the target itself as a feature.

This is invalid because the target is only known after observing April performance.

I compare an honest model with a deliberately leaked model, then remove the leaked feature.

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score


clean_data = model_data.dropna(
    subset=features + ["is_future_decline"]
).copy()


# Honest model
X = clean_data[features]
y = clean_data["is_future_decline"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42
)

honest_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

honest_model.fit(X_train, y_train)

honest_score = accuracy_score(
    y_test,
    honest_model.predict(X_test)
)

print("Honest score:", honest_score)

Honest score: 0.5936979665977092


In [7]:
# Deliberately add leakage
clean_data["label_leak"] = clean_data["is_future_decline"]

leaky_features = features + ["label_leak"]

X_leaky = clean_data[leaky_features]
y = clean_data["is_future_decline"]

X_train, X_test, y_train, y_test = train_test_split(
    X_leaky,
    y,
    test_size=0.25,
    random_state=42
)

leaky_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

leaky_model.fit(X_train, y_train)

leaky_score = accuracy_score(
    y_test,
    leaky_model.predict(X_test)
)

print("Honest score:", honest_score)
print("Leaky score:", leaky_score)


# Remove the leak
clean_data = clean_data.drop(columns=["label_leak"])

print(
    "Leak removed:",
    "label_leak" not in clean_data.columns
)

Honest score: 0.5936979665977092
Leaky score: 1.0
Leak removed: True


### Leakage result

The leaked model performs artificially well because `label_leak` directly contains the answer the model is supposed to predict.

This feature is not available at the prediction moment and is therefore invalid.

I removed it and keep only the five March features.

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

### Excluded fields

- `april_impressions` — excluded because it is future information.
- `is_future_decline` — excluded because it is the target, not a feature.
- `client_hash_id` — used only for grouping and joins, not as a model feature.
- `content_hash_id` — used only to identify content items, not as a model feature.
- `gsc_data_available` — used only to filter valid observations.
- Existing FlyRank flags or scores — excluded because they already encode product decisions and could create circular leakage.
- URLs and private queries — excluded for privacy and because they are not needed for this feature vector.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.